# Car damage detection: Colab training runner

Thin runner: all logic lives in `src/`. This notebook only clones the repo, prepares the data and calls `src/train.py`.

**Before running**
1. *Runtime > Change runtime type > T4 GPU*.
2. Add your Roboflow key as a Colab secret: key icon in the left sidebar > name `ROBOFLOW_API_KEY` > enable *Notebook access*.

Runs are saved to Google Drive. If the session disconnects, just run all cells again: finished experiments are skipped and an interrupted one resumes from its last epoch.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/car_damage_detection"  # runs + experiments.csv live here
!mkdir -p {DRIVE}

In [ ]:
import os

REPO_DIR = "/content/car_damage_detection"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/tatar01230/car_damage_detection.git {REPO_DIR}
%cd {REPO_DIR}
!git pull --ff-only
!git log --oneline -1
!pip install -q -r requirements.txt

In [ ]:
# Download the raw data and build the cleaned split (same scripts as locally)
import hashlib

from google.colab import userdata

os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
!python -m src.download_data
!python -m src.clean_data

# The cleaned split must be identical to the one built locally (reproducibility check)
EXPECTED_MANIFEST_MD5 = "3b3c99a6e9c158cec52be5a7b313dc3f"
md5 = hashlib.md5(open("data/clean/car-damage-v2/manifest.csv", "rb").read()).hexdigest()
assert md5 == EXPECTED_MANIFEST_MD5, f"Split differs from the local one: {md5}. Stop and investigate."
print("Split matches the local build:", md5)

In [ ]:
# Train. Finished experiments are skipped; an interrupted one resumes.
# Run them in order; trim this list to fit the session's GPU quota.
EXPERIMENTS = ["E1", "E2", "E3", "E4", "E5"]

for exp in EXPERIMENTS:
    !python -m src.train --exp configs/experiments/{exp}.yaml --project {DRIVE}/runs --results-csv {DRIVE}/experiments.csv

In [ ]:
import pandas as pd

pd.read_csv(f"{DRIVE}/experiments.csv")